# KLUE-RoBERTa 워크플로우 타입 분류 모델 학습

이 노트북은 KLUE-RoBERTa 모델을 사용하여 워크플로우 타입을 분류하는 모델을 학습합니다.

## 데이터셋
- 학습 세트: 약 350개
- 검증 세트: 약 45개
- 테스트 세트: 약 45개
- 총 샘플: 약 440개

## 학습 목표
2단계 분류: 워크플로우 타입 분류 (business_trip vs None)

## 클래스
- 0: business_trip (출장/휴가 워크플로우)
- 1: None (워크플로우가 아니거나 다른 타입)


## 1. 환경 설정 및 라이브러리 설치


In [ ]:
# 필수 라이브러리 설치
%pip install -q transformers datasets accelerate scikit-learn pandas numpy torch torchvision torchaudio


In [ ]:
# TPU 지원 라이브러리 설치 (코랩 TPU 사용 시)
try:
    import torch_xla
    print("torch_xla 이미 설치됨")
except ImportError:
    print("torch_xla 설치 중...")
    %pip install -q cloud-tpu-client==0.10 torch-xla[tpu]==2.0.0 -f https://storage.googleapis.com/libtpu-releases/index.html


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import json
from datetime import datetime


## 2. 환경 확인 (GPU/TPU)


In [ ]:
# 환경 확인
print("=" * 60)
print("환경 확인")
print("=" * 60)

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = "cuda"
else:
    print("GPU 사용 불가 - CPU로 학습 (매우 느림)")
    device = "cpu"

# TPU 확인
try:
    import torch_xla.core.xla_model as xm
    print(f"TPU 사용 가능: True")
    print(f"TPU 디바이스: {xm.xla_device()}")
    device = "tpu"
except ImportError:
    print("TPU 사용 불가 (torch_xla 미설치)")
except Exception as e:
    print(f"TPU 확인 실패: {e}")

print("=" * 60)


In [ ]:
# Google Drive 마운트 (데이터셋이 Drive에 있는 경우)
# from google.colab import drive
# drive.mount('/content/drive')

# 데이터셋 경로 설정
# 방법 1: 코랩에 직접 업로드한 파일 사용
data_dir = "data/step2_workflow_type_dataset"

# 방법 2: Google Drive에서 로드
# data_dir = "/content/drive/MyDrive/step2_workflow_type_dataset"

# 방법 3: GitHub에서 다운로드
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step2_workflow_type_dataset/train.csv
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step2_workflow_type_dataset/val.csv
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step2_workflow_type_dataset/test.csv
# data_dir = "."

# 데이터셋 로드
print("=" * 60)
print("데이터셋 로드")
print("=" * 60)

data_path = Path(data_dir)

# 파일 경로 확인
train_path = data_path / "train.csv"
val_path = data_path / "val.csv"
test_path = data_path / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"학습 데이터를 찾을 수 없습니다: {train_path}")
if not val_path.exists():
    raise FileNotFoundError(f"검증 데이터를 찾을 수 없습니다: {val_path}")

# CSV 로드
print(f"학습 데이터 로드: {train_path}")
train_df = pd.read_csv(train_path)

print(f"검증 데이터 로드: {val_path}")
val_df = pd.read_csv(val_path)

if test_path.exists():
    print(f"테스트 데이터 로드: {test_path}")
    test_df = pd.read_csv(test_path)
else:
    print("테스트 데이터 없음 - 검증 데이터로 대체")
    test_df = val_df.copy()

# 데이터 확인
print(f"\n학습 세트: {len(train_df)}개")
print(f"검증 세트: {len(val_df)}개")
print(f"테스트 세트: {len(test_df)}개")

# 클래스 분포
print(f"\n클래스 분포 (학습 세트):")
print(train_df['label'].value_counts().sort_index())
business_trip_count = (train_df['label'] == 0).sum()
none_count = (train_df['label'] == 1).sum()
print(f"business_trip (0): {business_trip_count}개 ({business_trip_count/len(train_df)*100:.1f}%)")
print(f"None (1): {none_count}개 ({none_count/len(train_df)*100:.1f}%)")
print("=" * 60)


In [ ]:
# 모델 및 토크나이저 로드
model_name = "klue/roberta-base"
print(f"모델 로드 중: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # business_trip vs None
    problem_type="single_label_classification"
)

print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print("모델 로드 완료")


## 5. 데이터셋 전처리 (토크나이징)


In [ ]:
# 데이터셋 전처리
print("=" * 60)
print("데이터셋 전처리")
print("=" * 60)

def tokenize_function(examples):
    return tokenizer(
        examples["query"],
        truncation=True,
        padding=True,
        max_length=128
    )

# DataFrame을 Dataset으로 변환
train_dataset = Dataset.from_pandas(train_df[["query", "label"]])
val_dataset = Dataset.from_pandas(val_df[["query", "label"]])
test_dataset = Dataset.from_pandas(test_df[["query", "label"]])

# 토크나이징
print("토크나이징 중...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("전처리 완료")
print("=" * 60)


## 6. 평가 메트릭 정의


In [ ]:
# 평가 메트릭 정의
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


## 7. 모델 학습 (TPU 최적화)


In [ ]:
# TPU 사용 여부 확인
use_tpu = device == "tpu"

# 학습 설정
output_dir = "./step2_workflow_type_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,  # 작은 데이터셋이므로 에포크 수 증가
    per_device_train_batch_size=16 if not use_tpu else 8,  # TPU는 배치 크기 조정
    per_device_eval_batch_size=16 if not use_tpu else 8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,  # 작은 데이터셋에 맞게 조정
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available() and device == "cuda",  # TPU는 fp16 미지원
    dataloader_num_workers=2 if not use_tpu else 0,  # TPU는 num_workers=0
    report_to="none",
)

# TPU 사용 시 추가 설정
if use_tpu:
    import torch_xla.core.xla_model as xm
    # TPU 디바이스 설정
    device = xm.xla_device()
    model = model.to(device)
    print(f"모델을 TPU로 이동: {device}")

# Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # 작은 데이터셋이므로 patience 증가
)

print("=" * 60)
print("모델 학습")
print("=" * 60)
print(f"\n학습 설정:")
print(f"  디바이스: {device}")
print(f"  학습 샘플 수: {len(train_dataset)}")
print(f"  검증 샘플 수: {len(val_dataset)}")
print(f"  배치 크기: {training_args.per_device_train_batch_size}")
print(f"  학습률: {training_args.learning_rate}")
print(f"  에포크 수: {training_args.num_train_epochs}")
print(f"  총 스텝 수: {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")
print(f"\n학습 시작...")

# 학습
trainer.train()

print("\n학습 완료!")
print("=" * 60)


In [ ]:
# 테스트 세트 평가
print("=" * 60)
print("모델 평가")
print("=" * 60)

print("테스트 세트 평가 중...")
test_results = trainer.evaluate(test_dataset)

print("\n=== 테스트 세트 성능 ===")
for key, value in test_results.items():
    if 'loss' not in key:
        print(f"{key}: {value:.4f}")

# 예측 및 혼동 행렬
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label'].values

print("\n=== 혼동 행렬 ===")
cm = confusion_matrix(true_labels, pred_labels)
print(cm)
print("\n(행: 실제, 열: 예측)")
print("0: business_trip, 1: None")
print("=" * 60)


## 9. 모델 저장


In [ ]:
# 모델 저장
output_dir = "./step2_workflow_type_final"

print("=" * 60)
print("모델 저장")
print("=" * 60)

# 모델 저장
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# 메타데이터 저장
metadata = {
    "model_name": "klue/roberta-base",
    "task": "workflow_type_classification",
    "num_labels": 2,
    "saved_at": datetime.now().isoformat(),
    "description": "2단계 워크플로우 타입 분류 모델 (business_trip vs None)",
    "classes": {
        "0": "business_trip",
        "1": "none"
    }
}

metadata_path = Path(output_dir) / "model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"모델 저장 완료: {output_dir}")

# 저장된 파일 목록
saved_files = list(Path(output_dir).glob("*"))
print(f"\n저장된 파일 ({len(saved_files)}개):")
for file in sorted(saved_files):
    if file.is_file():
        size = file.stat().st_size
        if size > 1024 * 1024:
            size_str = f"{size / (1024 * 1024):.2f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size} B"
        print(f"  - {file.name} ({size_str})")

print("=" * 60)


## 10. 모델 다운로드

학습된 모델을 다운로드하여 로컬에 설치하세요.


In [ ]:
# 방법 1: ZIP 파일로 다운로드
from google.colab import files
import zipfile
import os

# ZIP 파일 생성
output_dir = "./step2_workflow_type_final"
zip_path = "step2_workflow_type_model.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

print(f"ZIP 파일 생성 완료: {zip_path}")
print("다운로드 시작...")

# 다운로드
files.download(zip_path)

print("다운로드 완료!")


## 다음 단계

1. ✅ 모델 학습 완료
2. ✅ 모델 파일 다운로드
3. ⏳ 다운로드한 ZIP 파일 압축 해제
4. ⏳ 모든 파일을 `models/step2_workflow_type_final/` 디렉토리에 복사
5. ⏳ `agents/ml_workflow_type_classifier.py` 모듈 생성 및 통합

### 로컬 설치 방법

1. 다운로드한 `step2_workflow_type_model.zip` 압축 해제
2. 모든 파일을 프로젝트의 `models/step2_workflow_type_final/` 디렉토리에 복사:
   ```
   models/step2_workflow_type_final/
   ├── config.json
   ├── pytorch_model.bin 또는 model.safetensors
   ├── tokenizer_config.json
   ├── special_tokens_map.json
   ├── vocab.json
   └── merges.txt
   ```

### 예상 학습 시간

- **v5e-1 TPU**: 15-25분 (작은 데이터셋)
- **T4 GPU**: 20-30분
- **CPU**: 수 시간 (비권장)

### 주의사항

- 작은 데이터셋이므로 과적합에 주의
- Early Stopping이 활성화되어 있음
- 검증 세트 성능을 모니터링하여 최적 모델 선택
